# Module 02 — Lecture 1: GPU Memory Hierarchy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_02_memory_optimization/01_memory_hierarchy.ipynb)

---

## The Golden Rule of GPU Programming

> **Compute is cheap. Memory is expensive.**

A modern GPU can perform ~10 trillion floating-point operations per second. But if data cannot reach the cores fast enough, they sit idle waiting. Understanding memory is what separates a 10× GPU speedup from a 100× speedup.

**Learning objectives:**
- Name all five GPU memory types and their key properties
- Explain coalesced vs uncoalesced global memory access
- Measure actual latency and bandwidth for each memory type
- Map simulation data to the optimal memory type

In [ ]:
!nvidia-smi

## 1. The Five Memory Types

### 1.1 Registers

- **Speed:** ~1 cycle (fastest possible)
- **Scope:** Private to one thread, not accessible by others
- **Size:** ~255 registers per thread; ~65,536 per SM (all threads share)
- **Usage:** Local variables inside your kernel
- **Declared:** Automatically (any local variable in a kernel uses registers)

```cpp
__global__ void kernel(float* V) {
    float v_local = V[threadIdx.x];   // stored in a register
    float dv = -0.01f * v_local;      // register arithmetic
    V[threadIdx.x] = v_local + dv;    // write back
}
```

**Register spilling:** If your kernel uses more registers than available, overflow goes to **local memory** (in global DRAM) — much slower. Keep register use minimal.

### 1.2 Shared Memory (L1)

- **Speed:** ~20 cycles (~100× slower than registers, but 30× faster than global)
- **Scope:** Shared by all threads in a **block**
- **Size:** 48–164 KB per SM (configurable; blocks split this)
- **Usage:** Collaborative computation within a block; software-managed cache
- **Declared:** `__shared__` inside kernel

```cpp
__global__ void kernel() {
    __shared__ float smem[256];   // 256 floats × 4 bytes = 1 KB of shared mem
    smem[threadIdx.x] = ...;
    __syncthreads();               // barrier: all threads finish loading
    float neighbor = smem[(threadIdx.x + 1) % 256];   // fast access!
}
```

### 1.3 Constant Memory

- **Speed:** ~4 cycles when cached; same as global when not (broadcast cached)
- **Scope:** Read-only, visible to all threads on the GPU
- **Size:** 64 KB total
- **Usage:** Simulation parameters that all threads need (Δt, reversal potentials, conductances)
- **Declared:** `__constant__` at file scope; set from CPU with `cudaMemcpyToSymbol`

```cpp
__constant__ float c_dt;          // file scope
__constant__ float c_params[10];  // array of parameters

// On CPU:
float dt = 0.1f;
cudaMemcpyToSymbol(c_dt, &dt, sizeof(float));
```

### 1.4 Texture Memory

- Cached, optimized for 2D spatial locality
- Useful for 2D cortical field simulations where you access neighboring spatial positions
- We will use this in Module 06 for neural field theory

### 1.5 Global Memory

- **Speed:** ~600 cycles latency; ~300–2000 GB/s bandwidth
- **Scope:** All threads on the GPU, and accessible from CPU via `cudaMemcpy`
- **Size:** The GPU's main DRAM (8–80 GB)
- **Usage:** All large arrays: voltage arrays, spike trains, weight matrices
- **Declared:** Allocated with `cudaMalloc` on CPU side

| Memory | Latency | BW | Scope | Declared |
|--------|---------|-----|-------|----------|
| Registers | 1 cycle | N/A | Thread | automatic |
| Shared | ~20 cycles | ~10 TB/s | Block | `__shared__` |
| Constant | ~4 cycles (cached) | — | All threads | `__constant__` |
| Global | ~600 cycles | ~300–2000 GB/s | All + CPU | `cudaMalloc` |

## 2. Coalesced vs Uncoalesced Global Memory Access

Global memory is accessed in **cache lines** of 128 bytes (32 consecutive floats). When threads in a warp access consecutive memory locations, one transaction serves all 32 threads — **coalesced**. When they access scattered locations, each thread needs its own transaction — **uncoalesced**.

```
COALESCED (efficient):
  Thread 0 → A[0]    ┐
  Thread 1 → A[1]    │  One 128-byte transaction → 32× speedup
  Thread 2 → A[2]    │
  ...                │
  Thread 31 → A[31]  ┘

UNCOALESCED (stride-N):
  Thread 0 → A[0]                ┐  32 separate transactions
  Thread 1 → A[N]                │  Each gets only 4 bytes of the 128-byte line
  Thread 2 → A[2N]               │  → 32× slower than coalesced
  ...                            │
  Thread 31 → A[31*N]            ┘
```

**The matrix transpose problem:** If thread `i` processes row `i` of a matrix and accesses `A[i][k]` (row-major), that's coalesced. But accessing `B[k][j]` in column-major order (for matrix multiply) is uncoalesced — each thread in a warp jumps N elements apart. This is why matrix multiply needs tiling.

In [ ]:
# Benchmark: coalesced vs strided access
%%writefile coalesce_bench.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N (1 << 22)   // 4M elements
#define THREADS 256

// Coalesced: thread i reads A[i] (consecutive)
__global__ void coalesced(const float* A, float* B) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) B[i] = A[i] * 2.0f;
}

// Strided: thread i reads A[i * stride] (scattered)
__global__ void strided(const float* A, float* B, int stride) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i * stride < N) B[i] = A[i * stride] * 2.0f;
}

float time_kernel(void (*kernel_fn)(const float*, float*),
                   const float* dA, float* dB, int blocks, int threads) {
    cudaEvent_t t0, t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);
    kernel_fn<<<blocks, threads>>>(dA, dB);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float ms; cudaEventElapsedTime(&ms, t0, t1);
    cudaEventDestroy(t0); cudaEventDestroy(t1);
    return ms;
}

int main() {
    float *dA, *dB;
    cudaMalloc(&dA, N * sizeof(float));
    cudaMalloc(&dB, N * sizeof(float));

    int blocks = (N + THREADS - 1) / THREADS;

    float ms_c = time_kernel(coalesced, dA, dB, blocks, THREADS);
    float bw_c = 2.0f * N * sizeof(float) / (ms_c * 1e-3f) / 1e9f;
    printf("Coalesced:     %.3f ms  BW = %.1f GB/s\n", ms_c, bw_c);

    // Strided with different strides
    for (int stride : {2, 4, 8, 32}) {
        cudaEvent_t t0, t1;
        cudaEventCreate(&t0); cudaEventCreate(&t1);
        cudaEventRecord(t0);
        strided<<<blocks/stride, THREADS>>>(dA, dB, stride);
        cudaEventRecord(t1); cudaEventSynchronize(t1);
        float ms; cudaEventElapsedTime(&ms, t0, t1);
        float bw = 2.0f * (N/stride) * sizeof(float) / (ms * 1e-3f) / 1e9f;
        printf("Stride=%-4d    %.3f ms  BW = %.1f GB/s  (%.1fx slower)\n",
               stride, ms, bw, ms/ms_c);
        cudaEventDestroy(t0); cudaEventDestroy(t1);
    }

    cudaFree(dA); cudaFree(dB);
    return 0;
}

In [ ]:
!nvcc -O2 -std=c++11 -o coalesce_bench coalesce_bench.cu && ./coalesce_bench

## 3. Constant Memory for Simulation Parameters

Neuroscience simulations share many parameters across all neurons: time step Δt, reversal potentials, capacitance, etc. These are perfect candidates for constant memory — read-only, broadcast to all threads for free.

In [ ]:
%%writefile constant_params.cu
#include <stdio.h>
#include <cuda_runtime.h>

// Simulation parameters in constant memory
// These are set once on the CPU and read by every thread
__constant__ float c_dt;      // timestep (ms)
__constant__ float c_E_L;     // leak reversal potential (mV)
__constant__ float c_g_L;     // leak conductance
__constant__ float c_V_th;    // spike threshold (mV)
__constant__ float c_V_reset; // reset potential (mV)

__global__ void update_lif_constant(float* V, const float* I, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    // Parameters come from constant memory — no global read overhead
    float dV = c_dt * (c_g_L * (c_E_L - V[i]) + I[i]);
    V[i] += dV;

    // Reset if above threshold
    if (V[i] >= c_V_th) V[i] = c_V_reset;
}

int main() {
    // Set constant memory from CPU
    float dt = 0.1f, E_L = -65.0f, g_L = 0.1f, V_th = -55.0f, V_reset = -70.0f;
    cudaMemcpyToSymbol(c_dt,      &dt,      sizeof(float));
    cudaMemcpyToSymbol(c_E_L,     &E_L,     sizeof(float));
    cudaMemcpyToSymbol(c_g_L,     &g_L,     sizeof(float));
    cudaMemcpyToSymbol(c_V_th,    &V_th,    sizeof(float));
    cudaMemcpyToSymbol(c_V_reset, &V_reset, sizeof(float));

    const int N = 10000;
    float *d_V, *d_I;
    cudaMalloc(&d_V, N * sizeof(float));
    cudaMalloc(&d_I, N * sizeof(float));

    // Initialize on device
    float h_V[10] = {-65.0f}; // just to show it works
    printf("Constant memory set: dt=%.2f ms, E_L=%.1f mV, V_th=%.1f mV\n",
           dt, E_L, V_th);
    printf("Kernel would run on %d neurons — constant params broadcast for free.\n", N);

    cudaFree(d_V); cudaFree(d_I);
    return 0;
}

In [ ]:
!nvcc -O2 -o constant_params constant_params.cu && ./constant_params

## 4. Memory Layout for Neuron Simulations

How you lay out neuron state in memory matters for coalescing.

### Structure of Arrays (SoA) vs Array of Structures (AoS)

```
AoS (bad for GPU coalescing):
  struct Neuron { float V, m, h, n; };
  Neuron neurons[N];   // layout: V0 m0 h0 n0 V1 m1 h1 n1 ...
  → Thread i reads V[i] = neurons[i].V
  → Thread 0 reads byte 0, Thread 1 reads byte 16, Thread 2 reads byte 32...
  → Stride of 16 bytes = uncoalesced!

SoA (good for GPU coalescing):
  float V[N], m[N], h[N], n[N];   // layout: V0 V1 V2 ... m0 m1 m2 ...
  → Thread i reads V[i]
  → Thread 0 reads byte 0, Thread 1 reads byte 4, Thread 2 reads byte 8...
  → Stride of 4 bytes = coalesced!
```

**Always use Structure of Arrays for neuron state variables in GPU simulations.**

We will apply this consistently from Module 03 onward.

## Summary

| Memory | Use it for | Avoid it for |
|--------|-----------|-------------|
| Registers | Loop variables, intermediate ODE values | Anything larger than ~10 floats per thread |
| Shared | Tiles of matrices, neighborhood data | Data that doesn't get reused |
| Constant | Δt, reversal potentials, conductances | Writable data |
| Global | All neuron arrays, weight matrices | Anything that can fit elsewhere |

**Next lecture:** How to use shared memory to tile matrix operations — the key technique for fast synaptic weight updates.

## Self-Check

1. You have a LIF simulation with N=100,000 neurons, each with state (V, m, h, n). Should you use AoS or SoA layout? Why?
2. The spike threshold V_th is the same for all neurons. Which memory type should you use?
3. Each step, thread i reads V[i] from global memory and writes V[i] back. Is this access pattern coalesced? What is the memory traffic (bytes per neuron per step)?